# 📊 Análisis de Ventas — DataRetail Levante, S.L.

**Caso práctico:** Informe comercial completo con Python, Pandas y SQL  
**Base de datos:** `ventas.db` (SQLite)  
**Herramientas:** Python · Pandas · Matplotlib · SQLite3

---

## Índice
1. Conexión a la base de datos
2. KPIs generales del negocio
3. Análisis de clientes
4. Análisis de productos
5. Evolución temporal de ventas
6. Análisis por canal y forma de pago
7. Rentabilidad — Margen bruto
8. Conclusiones de negocio

---
## 1. Conexión a la base de datos

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Estilo visual
plt.rcParams.update({
    'figure.facecolor': '#f9f9f9',
    'axes.facecolor': '#f9f9f9',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11
})

# Conexión
conn = sqlite3.connect('../bases_datos/ventas.db')
print('✅ Conexión establecida con ventas.db')

---
## 2. KPIs generales del negocio

In [ ]:
# --- Clientes ---
df_clientes_total = pd.read_sql_query(
    'SELECT COUNT(*) AS TotalClientes FROM Clientes', conn)

# --- Productos ---
df_productos_kpi = pd.read_sql_query('''
    SELECT
        COUNT(*)                  AS TotalProductos,
        ROUND(AVG(PrecioVenta),2) AS PrecioMedio,
        MIN(PrecioVenta)          AS PrecioMinimo,
        MAX(PrecioVenta)          AS PrecioMaximo,
        SUM(Stock)                AS StockTotal
    FROM Productos
''', conn)

# --- Ventas ---
df_ventas_kpi = pd.read_sql_query('''
    SELECT
        COUNT(DISTINCT v.IdVenta) AS TotalVentas,
        ROUND(SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)), 2) AS FacturacionTotal,
        ROUND(
            SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)) /
            COUNT(DISTINCT v.IdVenta)
        , 2) AS TicketMedio
    FROM Ventas v
    JOIN LineasVenta lv ON v.IdVenta = lv.IdVenta
''', conn)

print('='*50)
print('  RESUMEN EJECUTIVO — DataRetail Levante, S.L.')
print('='*50)
print(f"  👤 Clientes registrados : {df_clientes_total.iloc[0,0]}")
print(f"  📦 Productos en catálogo: {df_productos_kpi['TotalProductos'].iloc[0]}")
print(f"  📦 Stock total          : {df_productos_kpi['StockTotal'].iloc[0]:,} uds")
print(f"  💰 Precio medio         : {df_productos_kpi['PrecioMedio'].iloc[0]:.2f} €")
print(f"  🛒 Total ventas         : {df_ventas_kpi['TotalVentas'].iloc[0]}")
print(f"  💶 Facturación total    : {df_ventas_kpi['FacturacionTotal'].iloc[0]:,.2f} €")
print(f"  🎫 Ticket medio/venta   : {df_ventas_kpi['TicketMedio'].iloc[0]:,.2f} €")
print('='*50)

---
## 3. Análisis de clientes

In [ ]:
# Clientes por provincia
df_prov = pd.read_sql_query('''
    SELECT Provincia, COUNT(*) AS NumClientes
    FROM Clientes
    GROUP BY Provincia
    ORDER BY NumClientes DESC
''', conn)

print('Clientes por provincia:')
print(df_prov.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(df_prov['Provincia'], df_prov['NumClientes'],
              color=['#2563EB','#3B82F6','#60A5FA','#93C5FD'])
ax.bar_label(bars, padding=3, fontweight='bold')
ax.set_title('Distribución de clientes por provincia', fontweight='bold', pad=12)
ax.set_ylabel('Nº de clientes')
ax.set_xlabel('Provincia')
plt.tight_layout()
plt.savefig('../imagenes/clientes_por_provincia.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 5 clientes por facturación
df_top_clientes = pd.read_sql_query('''
    SELECT
        c.NombreCliente,
        COUNT(DISTINCT v.IdVenta)                                                      AS NumPedidos,
        ROUND(SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)), 2)     AS TotalComprado,
        CASE
            WHEN SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)) >= 2000 THEN 'VIP'
            WHEN SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)) >= 1000 THEN 'Frecuente'
            ELSE 'Ocasional'
        END AS Segmento
    FROM Clientes c
    JOIN Ventas v       ON c.IdCliente = v.IdCliente
    JOIN LineasVenta lv ON v.IdVenta   = lv.IdVenta
    GROUP BY c.IdCliente
    ORDER BY TotalComprado DESC
    LIMIT 5
''', conn)

print('Top 5 clientes por facturación:')
print(df_top_clientes.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
colores = ['#1D4ED8','#2563EB','#3B82F6','#60A5FA','#93C5FD']
bars = ax.barh(df_top_clientes['NombreCliente'][::-1],
               df_top_clientes['TotalComprado'][::-1],
               color=colores[::-1])
ax.bar_label(bars, fmt='%.0f €', padding=4, fontweight='bold')
ax.set_title('Top 5 clientes — Facturación total (€)', fontweight='bold', pad=12)
ax.set_xlabel('Importe (€)')
plt.tight_layout()
plt.savefig('../imagenes/top_clientes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Análisis de productos

In [ ]:
# Top 5 productos que más ingresos generan
df_top_prod = pd.read_sql_query('''
    SELECT
        p.NombreProducto,
        SUM(lv.Cantidad)                                                               AS Unidades,
        ROUND(SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)), 2)     AS ImporteTotal
    FROM LineasVenta lv
    JOIN Productos p ON lv.IdProducto = p.IdProducto
    GROUP BY lv.IdProducto
    ORDER BY ImporteTotal DESC
    LIMIT 5
''', conn)

print('Top 5 productos por importe generado:')
print(df_top_prod.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(df_top_prod['NombreProducto'][::-1],
               df_top_prod['ImporteTotal'][::-1],
               color='#2563EB')
ax.bar_label(bars, fmt='%.0f €', padding=4, fontweight='bold')
ax.set_title('Top 5 productos — Ingresos generados (€)', fontweight='bold', pad=12)
ax.set_xlabel('Importe (€)')
plt.tight_layout()
plt.savefig('../imagenes/top_productos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Indicadores por categoría
df_cat = pd.read_sql_query('''
    SELECT
        cat.NombreCategoria,
        COUNT(*)                    AS NumProductos,
        ROUND(AVG(p.PrecioVenta),2) AS PrecioMedio,
        SUM(p.Stock)                AS StockTotal
    FROM Productos p
    JOIN Categorias cat ON p.IdCategoria = cat.IdCategoria
    GROUP BY cat.IdCategoria
    ORDER BY StockTotal DESC
''', conn)

print('Indicadores por categoría:')
print(df_cat.to_string(index=False))

---
## 5. Evolución temporal de ventas

In [ ]:
# Ventas mensuales
df_mensual = pd.read_sql_query('''
    SELECT
        strftime('%Y-%m', v.FechaVenta)                                                    AS Mes,
        COUNT(DISTINCT v.IdVenta)                                                           AS NumVentas,
        ROUND(SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)), 2)          AS FacturacionMes
    FROM Ventas v
    JOIN LineasVenta lv ON v.IdVenta = lv.IdVenta
    GROUP BY Mes
    ORDER BY Mes
''', conn)

print('Facturación mensual:')
print(df_mensual.to_string(index=False))

# Gráfico
fig, ax1 = plt.subplots(figsize=(10, 5))

meses_cortos = [m[-2:] + '/' + m[:4] for m in df_mensual['Mes']]

ax1.fill_between(meses_cortos, df_mensual['FacturacionMes'],
                 alpha=0.3, color='#2563EB')
ax1.plot(meses_cortos, df_mensual['FacturacionMes'],
         color='#2563EB', linewidth=2.5, marker='o', markersize=6, label='Facturación (€)')
ax1.set_ylabel('Facturación (€)', color='#2563EB')
ax1.tick_params(axis='y', labelcolor='#2563EB')

ax2 = ax1.twinx()
ax2.bar(meses_cortos, df_mensual['NumVentas'],
        alpha=0.25, color='#F59E0B', label='Nº ventas')
ax2.set_ylabel('Nº de ventas', color='#92400E')
ax2.tick_params(axis='y', labelcolor='#92400E')

ax1.set_title('Evolución mensual de ventas — 2025', fontweight='bold', pad=12)
ax1.set_xlabel('Mes')
plt.xticks(rotation=30)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('../imagenes/evolucion_mensual.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Análisis por canal y forma de pago

In [ ]:
# Canal de venta
df_canal = pd.read_sql_query('''
    SELECT
        v.CanalVenta,
        COUNT(DISTINCT v.IdVenta)                                                    AS NumVentas,
        ROUND(SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)),2)    AS FacturacionCanal
    FROM Ventas v
    JOIN LineasVenta lv ON v.IdVenta = lv.IdVenta
    GROUP BY v.CanalVenta
    ORDER BY FacturacionCanal DESC
''', conn)

print('Ventas por canal:')
print(df_canal.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colores = ['#2563EB','#3B82F6','#93C5FD']

axes[0].pie(df_canal['FacturacionCanal'], labels=df_canal['CanalVenta'],
            autopct='%1.1f%%', colors=colores, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Facturación por canal (€)', fontweight='bold')

axes[1].pie(df_canal['NumVentas'], labels=df_canal['CanalVenta'],
            autopct='%1.1f%%', colors=colores, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Nº de ventas por canal', fontweight='bold')

plt.suptitle('Análisis por canal de venta', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../imagenes/canal_ventas.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Rentabilidad — Margen bruto

In [ ]:
# Margen bruto global
df_margen = pd.read_sql_query('''
    SELECT
        ROUND(SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)), 2) AS IngresoTotal,
        ROUND(SUM(lv.Cantidad * p.PrecioCoste), 2)                                AS CosteTotal,
        ROUND(SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0))
              - SUM(lv.Cantidad * p.PrecioCoste), 2)                               AS MargenBruto,
        ROUND(
            (SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0))
             - SUM(lv.Cantidad * p.PrecioCoste))
            / SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)) * 100
        , 1) AS MargenPct
    FROM LineasVenta lv
    JOIN Productos p ON lv.IdProducto = p.IdProducto
''', conn)

print('Rentabilidad global del negocio:')
print(f"  Ingresos  : {df_margen['IngresoTotal'].iloc[0]:>10,.2f} €")
print(f"  Costes    : {df_margen['CosteTotal'].iloc[0]:>10,.2f} €")
print(f"  Margen    : {df_margen['MargenBruto'].iloc[0]:>10,.2f} €")
print(f"  Margen %  : {df_margen['MargenPct'].iloc[0]:>9.1f} %")

# Margen por categoría
df_margen_cat = pd.read_sql_query('''
    SELECT
        cat.NombreCategoria,
        ROUND(SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)), 2) AS Ingresos,
        ROUND(SUM(lv.Cantidad * p.PrecioCoste), 2)                                 AS Costes,
        ROUND(SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0))
              - SUM(lv.Cantidad * p.PrecioCoste), 2)                               AS Margen,
        ROUND(
            (SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0))
             - SUM(lv.Cantidad * p.PrecioCoste))
            / SUM(lv.Cantidad * lv.PrecioUnitario * (1 - lv.Descuento/100.0)) * 100
        , 1) AS MargenPct
    FROM LineasVenta lv
    JOIN Productos  p   ON lv.IdProducto  = p.IdProducto
    JOIN Categorias cat ON p.IdCategoria  = cat.IdCategoria
    GROUP BY cat.IdCategoria
    ORDER BY MargenPct DESC
''', conn)

print('\nMargen bruto por categoría:')
print(df_margen_cat.to_string(index=False))

---
## 8. Conclusiones de negocio

A partir del análisis realizado con SQL y Python sobre la base de datos de **DataRetail Levante, S.L.**, se extraen las siguientes conclusiones:

### 📌 Clientes
- La empresa cuenta con **12 clientes**, todos ubicados en la Comunidad Valenciana y Murcia.
- **Alicante concentra el 58 %** de la cartera (7 clientes).
- El cliente con mayor facturación es **Logística Sureste** (≈ 3.240 €).

### 📌 Productos
- El catálogo tiene **15 productos** en 5 categorías, con un precio medio de **142,61 €**.
- El **Portátil Lenovo ThinkBook** es el producto que más ingresos genera (≈ 4.602 €), aunque la **Licencia antivirus anual** es la más vendida en unidades (37 uds).

### 📌 Ventas
- Se han registrado **24 ventas** con una facturación total de **17.786,99 €**.
- El **mejor mes** fue junio 2025 (3.857 €); el peor, septiembre 2025 (1.001 €).
- El canal **Presencial** supera al Online en facturación (8.698 € vs 6.287 €).

### 📌 Rentabilidad
- El margen bruto del negocio supera el **55 %**, lo que indica una buena estructura de costes.
- La categoría **Software** presenta el mayor margen porcentual.

In [ ]:
# Cerrar conexión
conn.close()
print('✅ Análisis completado. Conexión cerrada.')